In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np

for _path in (Path.cwd(), *Path.cwd().parents):
    if (_path / "pyproject.toml").is_file():
        _root = str(_path)
        if _root not in sys.path:
            sys.path.insert(0, _root)
        break
else:
    raise RuntimeError("Could not find project root (pyproject.toml)")

import matplotlib.pyplot as plt

from analysis.gaussian_fit_cycle_analysis import (
    build_cell_receptive_fields_df,
    compute_aic_r2_correlation,
    outlier_points_summary_df,
    select_aic_r2_outlier_points,
)
from analysis.gaussian_fit_cycle_plots import (
    plot_aic_r2_correlation,
    plot_all_outlier_analysis,
    plot_nan_r2_gaussian_evolutions,
    save_figure,
)
from cycles.cycles_paths import (
    GAUSSIAN_EVOLUTION_PLOTS_DIR,
    PLOTS_DIR,
    RESULTS_DIR,
)


In [ ]:
GAUSSIANS_RESULTS_PATH = RESULTS_DIR / "gaussian_rf_fits.npz"
RAW_DATA_DIR = RESULTS_DIR
TRUNCATED_FILE_PATHS = [RESULTS_DIR/fn for fn in ["cycles_ratemaps_truncated_10.npz","cycles_ratemaps_truncated_10_20.npz","cycles_ratemaps_truncated_20_30.npz"]]
assert GAUSSIANS_RESULTS_PATH.is_file(), f"File {GAUSSIANS_RESULTS_PATH.resolve()} does not exist"

PLOTS_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
data = np.load(GAUSSIANS_RESULTS_PATH)
print("\n".join([f"{k}: {v.shape}" for k, v in data.items()]))


In [ ]:
df_cell_receptive_fields = build_cell_receptive_fields_df(GAUSSIANS_RESULTS_PATH)
df_cell_receptive_fields.head()


In [ ]:
THRESHOLD_ACTIVITY_MAX = 0.2


## Cells which at some have a NaN


In [ ]:
MAX_PLOTS = 12  # set None to plot every (cell, room) pair (cyclical order)

figures = plot_nan_r2_gaussian_evolutions(
    df_cell_receptive_fields,
    signal_max_threshold=THRESHOLD_ACTIVITY_MAX,
    max_plots=MAX_PLOTS,
	im_width=1.0
)

GAUSSIAN_EVOLUTION_PLOTS_DIR.mkdir(parents=True, exist_ok=True)

for cell_idx, room_id, fig in figures:
    out_path = (
        GAUSSIAN_EVOLUTION_PLOTS_DIR
        / f"gaussian_evolution_cell{cell_idx}_room{room_id}.pdf"
    )
    save_figure(fig, out_path, close=False)
    plt.show()
    plt.close(fig)
    print(f"Saved {out_path}")


## Correlations


In [ ]:
R2_THRESHOLD = 0.5


In [ ]:
# AIC R2 correlations
aic_r2_stats = compute_aic_r2_correlation(
    df_cell_receptive_fields,
    r2_threshold=R2_THRESHOLD,
)
df_aic_r2_correlation = aic_r2_stats.filtered_df
fig = plot_aic_r2_correlation(aic_r2_stats, r2_threshold=R2_THRESHOLD)
out_path = PLOTS_DIR / "aic_r2_correlation.pdf"
save_figure(fig, out_path, close=False)
plt.show()
plt.close(fig)
print(
    f"Saved {out_path} "
    f"(n={len(aic_r2_stats.r2):,}, r={aic_r2_stats.correlation:.3f})"
)


### Filter correlations based on mean activity over cycles


In [ ]:

aic_r2_stats_filtered = compute_aic_r2_correlation(
    df_cell_receptive_fields,
    r2_threshold=R2_THRESHOLD,
    signal_max_threshold=THRESHOLD_ACTIVITY_MAX,
)
df_aic_r2_correlation_filtered = aic_r2_stats_filtered.filtered_df
fig = plot_aic_r2_correlation(aic_r2_stats_filtered, r2_threshold=R2_THRESHOLD)
out_path = PLOTS_DIR / "aic_r2_correlation_filtered.pdf"
save_figure(fig, out_path, close=False)
plt.show()
plt.close(fig)
print(
    f"Saved {out_path} "
    f"(n={len(aic_r2_stats_filtered.r2):,}, r={aic_r2_stats_filtered.correlation:.3f})"
)


#### Outliers


In [ ]:
initial_raw_gt_data = None


In [ ]:
N_OUTLIERS_PER_GROUP = 5
R2_POOL_FRACTION = 0.05
OUTLIER_PLOTS_DIR = PLOTS_DIR / "outliers"
SAVE_OUTLIER_PLOTS = True
OUTLIER_IM_WIDTH = 1.0

with np.load(GAUSSIANS_RESULTS_PATH) as meta:
    n_gaussians = int(meta["n_gaussians"])

outlier_points = select_aic_r2_outlier_points(
    df_aic_r2_correlation_filtered,
    n_per_group=N_OUTLIERS_PER_GROUP,
    r2_threshold=R2_THRESHOLD,
    r2_pool_fraction=R2_POOL_FRACTION,
)
outlier_summary = outlier_points_summary_df(outlier_points)
print(
    f"Selected {len(outlier_points)} outlier points "
    f"(target {4 * N_OUTLIERS_PER_GROUP}; "
    f"top/bottom {R2_POOL_FRACTION:.0%} $R^2$ pool, AIC extremes)"
)
display(outlier_summary)

metric_figs, gaussian_figs, initial_raw_gt_data = plot_all_outlier_analysis(
    df_aic_r2_correlation_filtered,
    TRUNCATED_FILE_PATHS,
    outlier_points,
    ratemaps_by_visit=initial_raw_gt_data,
    n_gaussians=n_gaussians,
    im_width=OUTLIER_IM_WIDTH,
)

OUTLIER_PLOTS_DIR.mkdir(parents=True, exist_ok=True)

for point, fig in metric_figs:
    stem = (
        f"outlier_r2{point.r2_band}_aic{point.aic_extremum}_cell{point.cell_idx}"
        f"_room{point.room_id}_visit{point.visit_idx}_metric"
    )
    if SAVE_OUTLIER_PLOTS:
        save_figure(fig, OUTLIER_PLOTS_DIR / f"{stem}.pdf", close=False)
    plt.show()
    plt.close(fig)

for point, fig in gaussian_figs:
    stem = (
        f"outlier_r2{point.r2_band}_aic{point.aic_extremum}_cell{point.cell_idx}"
        f"_room{point.room_id}_visit{point.visit_idx}_gaussian"
    )
    if SAVE_OUTLIER_PLOTS:
        save_figure(fig, OUTLIER_PLOTS_DIR / f"{stem}.pdf", close=False)
    plt.show()
    plt.close(fig)
    if SAVE_OUTLIER_PLOTS:
        print(f"Saved {OUTLIER_PLOTS_DIR / stem}.pdf")
